# Detección de Placas Vehiculares con YOLOv11 + OCR

Pipeline completo: transfer learning con **YOLO11** (Ultralytics) para detectar la región de la placa, luego **EasyOCR** para leer el texto. Dataset descargado desde [Roboflow Universe](https://universe.roboflow.com).

## 1. Instalación de dependencias

In [2]:
# ultralytics incluye YOLO11; easyocr es el motor OCR; roboflow descarga el dataset anotado
!pip install -q ultralytics easyocr roboflow opencv-python

import torch
print(f"GPU disponible: {torch.cuda.is_available()} | Dispositivo: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

GPU disponible: True | Dispositivo: Tesla T4


## 2. Descarga del dataset (Roboflow)

Usa un dataset de detección de placas desde Roboflow en formato **YOLOv11**. Puedes sustituirlo por cualquier otro proyecto de [universe.roboflow.com](https://universe.roboflow.com) que tenga anotaciones de placas. Reemplaza `api_key` con tu clave personal.

In [3]:
from roboflow import Roboflow

# Reemplaza con tu API key de Roboflow (gratuita en roboflow.com)
rf = Roboflow(api_key="J1ecC83LAXy0J54fisWC")

# Dataset público de placas vehiculares — puedes cambiar workspace/project/version
project = rf.workspace("roboflow-universe-projects").project("license-plate-recognition-rxg4e")
version  = project.version(4)
dataset  = version.download("yolov11")   # formato compatible con YOLO11

import shutil, os
# Mover a /content/dataset para rutas consistentes en el resto del notebook
if os.path.exists("/content/dataset"):
    shutil.rmtree("/content/dataset")
shutil.move(dataset.location, "/content/dataset")
print("Dataset en:", "/content/dataset")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to License-Plate-Recognition-4 in yolov11:: 100%|██████████| 48488/48488 [00:06<00:00, 7306.96it/s] 


Dataset en: /content/dataset


## 3. Validación de la estructura del dataset

Verifica que las carpetas `train/`, `valid/` y `test/` existan con sus imágenes y etiquetas antes de entrenar.

In [4]:
import os

base = "/content/dataset"
splits = ["train/images", "train/labels", "valid/images", "valid/labels", "test/images", "test/labels"]

for s in splits:
    path = os.path.join(base, s)
    count = len(os.listdir(path)) if os.path.exists(path) else 0
    status = "✅" if count > 0 else "❌"
    print(f"{status} {s}: {count} archivos")

✅ train/images: 21173 archivos
✅ train/labels: 21173 archivos
✅ valid/images: 2046 archivos
✅ valid/labels: 2046 archivos
✅ test/images: 1019 archivos
✅ test/labels: 1019 archivos


## 4. Entrenamiento con Transfer Learning (YOLO11)

Cargamos `yolo11n.pt` (pesos preentrenados en COCO) y los ajustamos sobre el dataset de placas — esto es transfer learning: la red ya sabe detectar objetos generales, solo aprende a especializarse en placas.

In [ ]:
import os
from ultralytics import YOLO

# Intentar cargar el último checkpoint si existe para retomar
last_ckpt = "runs/detect/placas_yolo11/weights/last.pt"

if os.path.exists(last_ckpt):
    print(f"Retomando entrenamiento desde {last_ckpt}")
    model = YOLO(last_ckpt)
    resume = True
else:
    print("Iniciando entrenamiento desde yolo11n.pt")
    # Cargar YOLO11 nano preentrenado (más rápido; cambia a yolo11s.pt / yolo11m.pt para más precisión)
    model = YOLO("yolo11n.pt")
    resume = False

# TensorBoard para monitorear métricas durante el entrenamiento
%reload_ext tensorboard
%tensorboard --logdir runs/detect

# Entrenar: freeze=10 congela las primeras 10 capas para preservar features generales de COCO
# resume=True continuará desde la última época si se interrumpió
results = model.train(
    data="/content/dataset/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    freeze=10,       # transfer learning: congela backbone, entrena solo head
    patience=10,     # early stopping si no mejora en 10 épocas
    name="placas_yolo11",
    resume=resume
)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


<IPython.core.display.Javascript object>

Ultralytics 8.4.45 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=placas_yolo11, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=1

## 5. Evaluación del modelo

Mide precisión, recall y mAP50 sobre el conjunto de validación.

In [ ]:
# Cargar el mejor checkpoint guardado durante el entrenamiento
model_best = YOLO("runs/detect/placas_yolo11/weights/best.pt")

metrics = model_best.val(data="/content/dataset/data.yaml")

print(f"mAP50:    {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"Precision:{metrics.box.mp:.4f}")
print(f"Recall:   {metrics.box.mr:.4f}")

## 6. Pipeline OCR: detectar placa → leer texto

YOLO11 localiza la región de la placa (bounding box). EasyOCR lee los caracteres dentro de ese recorte. Se aplica preprocesamiento (escala ×2 + umbralización Otsu) para mejorar la tasa de reconocimiento.

In [ ]:
import cv2
import easyocr
import numpy as np
from IPython.display import display
import PIL.Image as Image

# Inicializar OCR una sola vez (carga el modelo en memoria)
reader = easyocr.Reader(['en'], gpu=torch.cuda.is_available())

def detect_and_read(image_path, conf=0.5):
    """Detecta placas con YOLO11 y extrae texto con EasyOCR."""
    results = model_best.predict(image_path, conf=conf, imgsz=640, verbose=False)
    img = cv2.imread(image_path)
    output = img.copy()
    plates_found = []

    for box in results[0].boxes.xyxy.cpu().numpy():
        x1, y1, x2, y2 = map(int, box)
        crop = img[y1:y2, x1:x2]

        # Escalar ×2 mejora la resolución para OCR en placas pequeñas
        crop = cv2.resize(crop, None, fx=2, fy=2, interpolation=cv2.INTER_CUBIC)

        # Convertir a gris y umbralizar (Otsu binariza adaptativamente según el histograma)
        gray  = cv2.cvtColor(crop, cv2.COLOR_BGR2GRAY)
        _, thresh = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

        # EasyOCR sobre la imagen binarizada; paragraph=True combina palabras en una sola línea
        texts = reader.readtext(thresh, detail=0, paragraph=True)
        plate_text = ' '.join(texts).strip()
        plates_found.append({'bbox': (x1, y1, x2, y2), 'text': plate_text})

        # Anotar imagen de salida con bbox y texto
        cv2.rectangle(output, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(output, plate_text, (x1, y1 - 8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 255, 0), 2)

    display(Image.fromarray(cv2.cvtColor(output, cv2.COLOR_BGR2RGB)))
    return plates_found

# --- Prueba con una imagen del conjunto de test ---
test_dir = "/content/dataset/test/images"
sample   = os.path.join(test_dir, os.listdir(test_dir)[0])
plates   = detect_and_read(sample)

for p in plates:
    print(f"Placa detectada: '{p['text']}'  |  bbox: {p['bbox']}")

## 7. Inferencia en lote sobre todo el conjunto de test

In [ ]:
# Procesar todas las imágenes de test y recopilar resultados
all_results = []

for fname in sorted(os.listdir(test_dir)):
    if not fname.lower().endswith(('.jpg', '.jpeg', '.png')):
        continue
    path   = os.path.join(test_dir, fname)
    plates = detect_and_read(path, conf=0.5)
    all_results.append({'file': fname, 'plates': plates})
    for p in plates:
        print(f"{fname}  →  '{p['text']}'")

## 8. Exportar modelo

Exporta el modelo entrenado a **ONNX** (compatible con cualquier runtime) y descarga los pesos para usarlos fuera de Colab.

In [ ]:
from google.colab import files

# Exportar a ONNX para deploy en producción (TensorRT, OpenVINO, etc.)
model_best.export(format="onnx", opset=12)

# Comprimir y descargar pesos + ONNX
weights_dir = "runs/detect/placas_yolo11/weights"
shutil.make_archive("/content/placas_yolo11_weights", 'zip', weights_dir)
files.download("/content/placas_yolo11_weights.zip")

## Exportar todos los formatos y copiar a Google Drive

El siguiente bloque intenta exportar el modelo a varios formatos (ONNX, TorchScript, SavedModel, TFLite), recopila los archivos en /content/modelos, los comprime y luego los copia a Google Drive si está montado; en caso contrario, ofrecerá la descarga directa.

In [ ]:
# Exportar varios formatos, agrupar y copiar a Google Drive (o descargar)
import os, shutil, glob
from google.colab import files

# Seleccionar el objeto modelo a exportar (usar model_best si existe)
try:
    model_to_export = model_best
except NameError:
    model_to_export = globals().get('model', None)

if model_to_export is None:
    print('No hay ningún modelo cargado en model_best ni model. Ejecuta o carga el modelo antes de exportar.')
else:
    out_dir = '/content/modelos'
    os.makedirs(out_dir, exist_ok=True)

    formats = ['onnx', 'torchscript', 'saved_model', 'tflite']
    for fmt in formats:
        try:
            print(f'Exportando a {fmt}...')
            model_to_export.export(format=fmt)
        except Exception as e:
            print(f'Fallo al exportar {fmt}:', e)

    # Buscar archivos exportados y copiarlos a out_dir
    candidates = [
        'runs/detect/placas_yolo11/weights',
        'runs/detect/placas_yolo11',
        '.'
    ]

    exts = ['*.onnx', '*.torchscript', '*.tflite', '*.pb', '*.pt', '*.zip', '*.saved_model']
    for base in candidates:
        if not os.path.exists(base):
            continue
        for pattern in exts:
            for fpath in glob.glob(os.path.join(base, pattern)):
                try:
                    shutil.copy(fpath, out_dir)
                    print('Copiado:', fpath)
                except Exception as e:
                    print('No se pudo copiar', fpath, e)

    # Crear zip con todos los artefactos exportados
    archive_path = '/content/placas_yolo11_exports'
    shutil.make_archive(archive_path, 'zip', out_dir)
    zip_file = archive_path + '.zip'
    print('Archivo creado:', zip_file)

    # Intentar copiar a Google Drive si está disponible, sino ofrecer descarga
    try:
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
        dest = '/content/drive/MyDrive/placas_yolo11_exports.zip'
        shutil.copy(zip_file, dest)
        print('Copia completada a Google Drive:', dest)
    except Exception as e:
        print('No se pudo copiar a Google Drive (no montado o error):', e)
        try:
            files.download(zip_file)
        except Exception as e2:
            print('Descarga automática falló. Archivo ubicado en:', zip_file)